In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
path_to_test = "data/original_data/data/customer_clv_test.csv"
path_to_train = "data/original_data/data/customer_clv_train.csv"
path_to_transactions = "data/original_data/data/transactions_2016_2017.csv"

In [ ]:
train = pd.read_csv(path_to_train)
test = pd.read_csv(path_to_test)
transactions = pd.read_csv(path_to_transactions)

print("Dimensions: train:", train.shape, " test:", test.shape, 'transactions:', transactions.shape)
print("Train columns:", train.columns.to_list())
print("Transaction columns:", transactions.columns.to_list())

In [ ]:
train.head()

In [ ]:
transactions.head()

In [ ]:
transactions["order_date"] = pd.to_datetime(transactions["order_date"], errors='coerce')
transactions['pack_date'] = pd.to_datetime(transactions["pack_date"], errors='coerce')

print("missing values in order_date:", transactions["order_date"].isna().sum())
print("missing values in pack_date:", transactions['pack_date'].isna().sum())

In [ ]:
TARGET_VAR = "revenue_2018_2019"

y = pd.to_numeric(train[TARGET_VAR].dropna())
print('rows :', len(y), "| percent of O:", round(y.eq(0).mean() * 100,2))
print(y.describe(percentiles=[.5,.9,.95,.99]))

plt.figure()
plt.hist(y, bins=60)
plt.title(f"Target distribution: {TARGET_VAR}")
plt.xlabel(TARGET_VAR); plt.ylabel("count")
plt.tight_layout(); plt.show()

plt.figure()
plt.hist(np.log1p(y), bins=60)
plt.title(f"Target distribution (log1p): {TARGET_VAR}")
plt.xlabel(f"log1p({TARGET_VAR})"); plt.ylabel("count")
plt.tight_layout(); plt.show()



In [ ]:
print(np.log1p(y).describe(percentiles=[.5,.9,.95,.99]))

In [ ]:
print(len(transactions['prod_brand'].unique()))

In [ ]:
for column in transactions.columns.tolist():
    print("Missing values in", column, 'is', transactions[column].isna().sum(), " and percentage of the column missing", round(transactions[column].isna().mean() * 100, 2))

In [ ]:
for column in transactions.columns.to_list():
    print('Descriptive statistics for', column, ':', transactions[column].describe())

In [ ]:
for column in ["prod_clasp","prod_comfort_wear","prod_comfort_sole","prod_print","prod_insole","prod_heel","prod_type_4"]:
    transactions.drop(column, axis=1, inplace=True)

In [ ]:
## for returned_to_shop_id , prod_type_5 , prod_material NANs ???
for column in ["prod_type_5","prod_material","returned_to_shop_id"]:
    transactions[column].fillna('missing', inplace=True)

## still relevant to keep this feature with so many NaNs ??

In [ ]:
## duplicates
count= transactions.duplicated().sum()
percentage= round(count/len(transactions)*100,2)
print("Duplicated rows:",count, "percentage:", percentage )
## <1% so no problem

In [ ]:
for column in ["prod_season","prod_color","prod_type_1","prod_type_3","prod_type_5","prod_material"]:
    print("Unique for" ,column, sorted(transactions[column].unique()))

In [ ]:
print("Unique prod_brand:" , sorted(transactions["prod_brand"].unique()))

In [ ]:
transactions["prod_brand"]=transactions['prod_brand'].replace('Bana & Co', 'Bana')
transactions["prod_brand"]=transactions['prod_brand'].replace('Brako Anatomics', 'Brako')
transactions["prod_brand"]=transactions['prod_brand'].replace('Gabor Kids', 'Gabor')
transactions["prod_brand"]=transactions['prod_brand'].replace('Ragazzi Sport', 'Ragazzi')

In [ ]:
for column in ["sale_discount_applied", "sale_revenue", "prod_size"]:

    transactions[column]= pd.to_numeric(transactions[column],errors='coerce')
    transactions[column].dropna()
    
    plt.figure(figsize=(10, 5))
    sns.boxplot(y=transactions[column])
    plt.title(f'Outliers in {column}')
    plt.show()


    Q1=transactions[column].quantile(0.25)
    Q3=transactions[column].quantile(0.75)
    IQR= Q3-Q1
    count= ((transactions[column]<Q1-1.5*IQR) | (transactions[column]>Q3+1.5*IQR)).sum()
    percentage= round(count/len(transactions) *100 ,2)
    print("number of outliers:",count, ", percentage:", percentage)

In [ ]:
transactions['prod_size_bin'] = pd.cut(transactions['prod_size'], bins=[0, 35, 44, 100], labels=[1, 2, 3]) #1=kids, 2=standard, 3=big size
print(transactions['prod_size_bin'].value_counts())

In [ ]:
print((transactions["sale_revenue"]<0).sum())

print((transactions[transactions["sale_revenue"]<0]).head())

In [ ]:
transactions['discount_quantile'] = pd.qcut(transactions['sale_discount_applied'], q=5, labels=['Q1 (0-20%)', 'Q2 (20-40%)', 'Q3 (40-60%)', 'Q4 (60-80%)', 'Q5 (80-100%)'])
## Q1 = small discount, Q5= huge discount

transactions['revenue_quantile'] = pd.qcut(transactions['sale_revenue'], q=5,labels=['Q1 (0-20%)', 'Q2 (20-40%)', 'Q3 (40-60%)', 'Q4 (60-80%)', 'Q5 (80-100%)'])
## Q1 = small revenue, Q5= huge revenue


In [ ]:
transactions.head()

In [ ]:
train.head()

In [ ]:
#DORIAN
## Construction of the train dataset before splitting it.
df = transactions.merge(train, on="cust_id", how="inner")

print(df.shape)
df.head()

In [ ]:
customers = train["cust_id"]
y_binary = (train["revenue_2018_2019"] > 0).astype(int)

cust_cal, cust_val = train_test_split(
    customers,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)
X_cal = df[df["cust_id"].isin(cust_cal)]
X_val = df[df["cust_id"].isin(cust_val)]

In [ ]:
#Quick sanitity ceck
set(X_cal["cust_id"]).intersection(set(X_val["cust_id"]))
print("Original zero rate:", (train["revenue_2018_2019"] == 0).mean())

print(
    "Calibration zero rate:",
    X_cal.groupby("cust_id")["revenue_2018_2019"].first().eq(0).mean()
)

print(
    "Validation zero rate:",
    X_val.groupby("cust_id")["revenue_2018_2019"].first().eq(0).mean()
)

In [ ]:
X_cal.to_csv("data/calibration_transactions.csv", index=False)
X_val.to_csv("data/validation_transactions.csv", index=False)
#Run this to create those dataset and instead of just having them in your kernel.